In [4]:
from dotenv import load_dotenv
load_dotenv()
import os
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")



In [5]:
#Deterministic approach - rule based
def deterministic_guardrail(text: str) -> bool:
    """Returns True if content is blocked."""
    banned_keywords = ["hack", "exploit", "malware", "bomb"]
    return any(kw in text.lower() for kw in banned_keywords)

test_inputs = [
    "How do I hack into a database?",
    "What is the capital of France?",
    "Explain how malware spreads",
]

for i in test_inputs:
    blocked=deterministic_guardrail(i)
    status="BLOCKED"if blocked else "ALLOWED"
    print(status)

BLOCKED
ALLOWED
BLOCKED


In [15]:
#model based approach - uses LLM for semantic understanding
from langchain.chat_models import init_chat_model

def model_based_guardrail(text: str) -> str:
    """Uses an LLM to evaluate content safety. Returns SAFE or UNSAFE."""
    model = init_chat_model(model="groq:qwen/qwen3-32b", temperature=0)
    prompt = f"""Is the following user input safe to process? 
Reply with only 'SAFE' or 'UNSAFE'.

Input: {text}"""
    result=model.invoke([{"role":"user","content":prompt}])
    return result.content.strip()

for i in test_inputs:
    blocked=model_based_guardrail(i)
    status="BLOCKED"if  "UNSAFE" in blocked else "ALLOWED"
    print(status)

BLOCKED
ALLOWED
ALLOWED


### PII- Personally Identifiable Information. 
redact, mask, hash, block

In [16]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_core.tools import tool

@tool
def customer_lookup(query: str) -> str:
    """Look up customer information."""
    return f"Customer record found for query: {query}"

agent=create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[customer_lookup],
    middleware=[
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True
        ),
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True
        ),
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True
        )
    ]
)
print("Agent with PII middleware created successfully!")

Agent with PII middleware created successfully!


In [17]:
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is john.doe@example.com and my card is 5105-1051-0510-5100. Can you help me?"
    }]
})

print(result["messages"][-1].content)

We found a customer record associated with the email [REDACTED_EMAIL]. Would you like me to display the customer details or assist with another request?


In [18]:
result

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and my card is ****-****-****-5100. Can you help me?', additional_kwargs={}, response_metadata={}, id='214c5b35-ffc1-4531-bb94-c29b4653266d'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user provided their email and a partial credit card number. They want help, but I need to figure out what they need. First, I should check if there\'s a function available to look up customer info. The tools provided include a customer_lookup function that takes a query parameter. Since the user gave their email and card number, maybe I can use that to look up their account. But wait, the card number is partially hidden. Do I need the full number? The function\'s parameters just say "query," so maybe I can use the email or the last four digits of the card. The email is redacted, but maybe the system can still process it. I should call the customer_lookup function with the email they provided, eve

In [19]:
try:
    result = agent.invoke({
        "messages": [{
            "role": "user",
            "content": "Here is my key: sk-abcdefghijklmnopqrstuvwxyz123456"
        }]
    })
    
except Exception as e:
    print(f"Blocked: {e}")

Blocked: Detected 1 instance(s) of api_key in text content


In [20]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.tools import tool

@tool
def search_web(query: str) -> str:
    """Search the web for information."""
    return f"Search results for: {query}"

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {to} with subject: {subject}"

@tool
def delete_records(table: str, condition: str) -> str:
    """Delete records from the database."""
    return f"Deleted records from {table} where {condition}"

hitl_agent = create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[search_web, send_email, delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,     
                "delete_records": True,  
                "search_web": False,      
            }
        ),
    ],
    checkpointer=InMemorySaver(),  # Required for state persistence
)
print("Human in the loop agent created")

Human in the loop agent created


In [21]:
config={"configurable":{"thread_id":"s_1"}}

result=hitl_agent.invoke(
    {"messages":[{"role":"user","content":"Send an email to team@company.com about the Q4 results"}]},
    config=config
)

print("Agent paused")
print(result)

Agent paused
{'messages': [HumanMessage(content='Send an email to team@company.com about the Q4 results', additional_kwargs={}, response_metadata={}, id='c0c3badf-9194-4c76-9c3e-a209da71fd0a'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants to send an email to team@company.com regarding the Q4 results. Let me check which function to use. The available functions are search_web, send_email, and delete_records. The send_email function seems appropriate here.\n\nThe send_email function requires to, subject, and body. The user provided the email address (team@company.com) and the topic (Q4 results). I need to infer the subject and body since they aren\'t specified. A common subject for such emails could be "Q4 Results Summary" or similar. The body should mention that the email is regarding the Q4 results, perhaps asking for a meeting or sharing the results. I\'ll draft a simple body like "Please find attached the Q4 results report. Let\'s discuss the det

In [22]:
approved_result=hitl_agent.invoke(
    Command(resume={"decisions":[{"type":"approve"}]}),
    config=config
)

print("Approoved")
print(approved_result["messages"][-1].content)

Approoved
The email has been successfully sent to team@company.com with the subject "Q4 Results Summary". Let me know if you'd like to follow up with any additional actions!


### Custom guardrail, custom middleware, before agent input

In [24]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool

class ContentFilterMiddleware(AgentMiddleware):
    """
    Deterministic guardrail: Block requests containing banned keywords.
    This runs BEFORE the agent processes anything — zero LLM cost for blocked requests.
    """

    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_message = state["messages"][0]
        if first_message.type != "human":
            return None

        content = first_message.content.lower()

        for keyword in self.banned_keywords:
            if keyword in content:
                print(f" Blocked — keyword detected: '{keyword}'")
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I cannot process requests containing inappropriate content. "
                            "Please rephrase your request."
                        )
                    }],
                    "jump_to": "end"
                }
        return None


@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"


# Create agent with content filter
filtered_agent = create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[search_tool],
    middleware=[
        ContentFilterMiddleware(
            banned_keywords=["hack", "exploit", "malware", "jailbreak", "bypass"]
        ),
    ],
)

print("Content filter agent created!")

Content filter agent created!


In [25]:
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "What is machine learning?"}]
})
print(result["messages"][-1].content)

Machine learning is a subset of artificial intelligence (AI) that focuses on developing algorithms and models that enable computers to learn patterns and make decisions from data without being explicitly programmed for specific tasks. These systems improve their performance over time as they are exposed to more data.

### Key Concepts:
1. **Learning from Data**: Instead of relying on fixed instructions, machine learning models identify patterns in historical or training data to make predictions or decisions.
2. **Types of Learning**:
   - **Supervised Learning**: Uses labeled data (e.g., "this image is a cat") to train models for tasks like classification (e.g., spam detection) or regression (e.g., predicting house prices).
   - **Unsupervised Learning**: Analyzes unlabeled data to find hidden patterns, such as clustering similar customers or reducing data complexity.
   - **Reinforcement Learning**: Involves an agent learning to make decisions by trial and error, guided by rewards or 

In [27]:
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "How do I hack into a server?"}]
})
print(result["messages"][-1].content)

 Blocked — keyword detected: 'hack'
I cannot process requests containing inappropriate content. Please rephrase your request.


### Custom Guardrail — After-Agent Hook (Output Safety)

In [29]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain_core.messages import AIMessage
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain_core.tools import tool

class SafetyGuardrailMiddleware(AgentMiddleware):
    """
    Model-based guardrail: Use an LLM to evaluate response safety.
    Runs AFTER the agent produces a response, before it reaches the user.
    """

    def __init__(self):
        super().__init__()
        # Use a smaller, cheaper model for the safety check
        self.safety_model = ChatGroq(model="qwen/qwen3-32b", temperature=0)

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        # Use a lightweight model as the safety judge
        safety_prompt = f"""Evaluate if this AI response is safe and appropriate for users.
Respond with only 'SAFE' or 'UNSAFE'.

Response to evaluate:
{last_message.content}"""

        result = self.safety_model.invoke([{"role": "user", "content": safety_prompt}])

        if "UNSAFE" in result.content.upper():
            print("⚠️  Output flagged as UNSAFE — replacing with safe fallback")
            last_message.content = (
                "I'm unable to provide that response. "
                "Please rephrase your request or contact support."
            )

        return None


@tool
def general_tool(query: str) -> str:
    """A general purpose tool."""
    return f"Tool result: {query}"


safe_agent = create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[general_tool],
    middleware=[SafetyGuardrailMiddleware()],
)

print("Output safety agent created!")

Output safety agent created!


### Layered / Combined Guardrails

In [30]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool

@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Search results: {query}"

@tool
def send_email_tool(to: str, body: str) -> str:
    """Send an email."""
    return f"Email sent to {to}"

# Full layered guardrail stack
production_agent = create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[search_tool, send_email_tool],
    middleware=[
        # Layer 1: Deterministic input filter (before agent)
        ContentFilterMiddleware(banned_keywords=["hack", "exploit", "malware"]),

        # Layer 2: PII redaction on input
       
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),

        # Layer 3: Human approval for sensitive tools
        HumanInTheLoopMiddleware(
            interrupt_on={"send_email_tool": True, "search_tool": False}
        ),

        # Layer 4: PII redaction on output
        PIIMiddleware("email", strategy="redact", apply_to_output=True),

        # Layer 5: Model-based output safety
        SafetyGuardrailMiddleware(),
    ],
    checkpointer=InMemorySaver(),
)

